In [ ]:
!pip install -q transformers accelerate bitsandbytes pillow gradio peft pandas matplotlib evaluate nltk tqdm
import os, json, torch, pandas as pd, matplotlib.pyplot as plt
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from transformers import Blip2Processor, Blip2ForConditionalGeneration, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from google.colab import drive
from tqdm.notebook import tqdm
import evaluate, nltk

# Academic Metrics Initialization
nltk.download('punkt')
bleu_metric = evaluate.load("bleu")
drive.mount('/content/drive')
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Project initialized on {device.upper()}")

In [ ]:
model_id = "Salesforce/blip2-opt-2.7b"
processor = Blip2Processor.from_pretrained(model_id)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

model = Blip2ForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

# Apply LoRA for Parameter Efficient Fine Tuning (PEFT)
model = prepare_model_for_kbit_training(model)
config = LoraConfig(
    r=16, lora_alpha=32, target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05, bias="none", task_type="CAUSAL_LM"
)
model = get_peft_model(model, config)
model.enable_input_require_grads() # Fix for the 'Gradients will be None' error
print("Model loaded with 4-bit quantization and LoRA Adapters.")

In [ ]:
class FlickrDataset(Dataset):
    def __init__(self, jsonl_path, img_dir, processor, limit=1000):
        with open(jsonl_path, 'r') as f:
            self.data = [json.loads(line) for line in f][:limit] # Subset for speed
        self.img_dir, self.processor = img_dir, processor

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        image = Image.open(os.path.join(self.img_dir, item['image_path'])).convert("RGB")
        text = f"Question: {item['question']} Answer:" if 'question' in item else item['caption']
        encoding = self.processor(images=image, text=text, padding="max_length", truncation=True, max_length=70, return_tensors="pt")
        return {k: v.squeeze(0) for k, v in encoding.items()}, text

# Config Paths (Adjust these to your Drive structure)
IMAGE_DIR = '/content/drive/MyDrive/Flickr8k/Images'
TRAIN_PATH = '/content/drive/MyDrive/Flickr8k/jsonl/train.jsonl'
VAL_PATH   = '/content/drive/MyDrive/Flickr8k/jsonl/val.jsonl'

# Using a subset of 1000 for Train and 200 for Val to finish in ~20 mins
train_loader = DataLoader(FlickrDataset(TRAIN_PATH, IMAGE_DIR, processor, limit=1000), batch_size=4, shuffle=True)
val_loader   = DataLoader(FlickrDataset(VAL_PATH, IMAGE_DIR, processor, limit=200), batch_size=4)

In [ ]:
def validate(model, dataloader):
    model.eval()
    val_loss, preds, refs = 0, [], []
    with torch.no_grad():
        for batch, texts in dataloader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch, labels=batch["input_ids"])
            val_loss += outputs.loss.item()

            # Generate for accuracy check
            gen_ids = model.generate(pixel_values=batch['pixel_values'], max_new_tokens=20)
            preds.extend(processor.batch_decode(gen_ids, skip_special_tokens=True))
            refs.extend([[t] for t in texts])

    bleu_score = bleu_metric.compute(predictions=preds, references=refs)['bleu']
    return val_loss / len(dataloader), bleu_score

def train_and_plot(epochs=3):
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
    scaler = torch.amp.GradScaler('cuda')
    history = {"train_loss": [], "val_loss": [], "val_bleu": []}

    for epoch in range(epochs):
        model.train()
        t_loss = 0
        loop = tqdm(train_loader, desc=f"Epoch {epoch+1}")
        for batch, _ in loop:
            optimizer.zero_grad()
            batch = {k: v.to(device) for k, v in batch.items()}
            with torch.amp.autocast(device_type='cuda'):
                loss = model(**batch, labels=batch["input_ids"]).loss
            scaler.scale(loss).backward()
            scaler.step(optimizer); scaler.update()
            t_loss += loss.item()
            loop.set_postfix(loss=loss.item())

        v_loss, v_bleu = validate(model, val_loader)
        history["train_loss"].append(t_loss/len(train_loader))
        history["val_loss"].append(v_loss)
        history["val_bleu"].append(v_bleu)
        print(f"Val Loss: {v_loss:.4f} | Val BLEU: {v_bleu:.4f}")

    # Plotting for Presentation
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1); plt.plot(history["train_loss"], 'r', label="Train"); plt.plot(history["val_loss"], 'b', label="Val")
    plt.title("Loss Curves"); plt.legend(); plt.grid(True)
    plt.subplot(1, 2, 2); plt.plot(history["val_bleu"], 'g-s', label="BLEU Accuracy")
    plt.title("Accuracy (BLEU-4)"); plt.grid(True)
    plt.savefig("metrics.png"); plt.show()
    return history

history = train_and_plot(epochs=3)

In [ ]:
import matplotlib.pyplot as plt
import gradio as gr

# 1. GENERATE THE GRAPH FROM YOUR ACTUAL TRAINING DATA
def generate_final_report(train_losses, val_bleus):
    plt.style.use('ggplot')
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

    # Loss Plot (Your values: 0.379, 0.327, and the upcoming 3rd value)
    ax1.plot(range(1, len(train_losses) + 1), train_losses, 'r-o', linewidth=2, label="Train Loss")
    ax1.set_title("Training Loss Convergence", fontsize=14, fontweight='bold')
    ax1.set_xlabel("Epochs")
    ax1.set_ylabel("Loss")
    ax1.grid(True, linestyle='--')
    ax1.legend()

    # BLEU Plot (Your values: 0.1067, and the improvement)
    ax2.plot(range(1, len(val_bleus) + 1), val_bleus, 'b-s', linewidth=2, label="Validation BLEU")
    ax2.set_title("Model Accuracy (BLEU-4)", fontsize=14, fontweight='bold')
    ax2.set_xlabel("Epochs")
    ax2.set_ylabel("Score")
    ax2.grid(True, linestyle='--')
    ax2.legend()

    plt.tight_layout()
    plt.savefig("final_metrics.png")
    plt.show()

# If your history variable is named 'history', run this:
# If you didn't save it to 'history', you can manually enter your numbers here:
# Example: generate_final_report([0.379, 0.327, 0.295], [0.1067, 0.125, 0.138])
try:
    generate_final_report(history["train_loss"], history["val_bleu"])
except:
    print("Please manually enter your loss/bleu values in the generate_final_report function.")

# 2. LAUNCH THE GRADIO UI
def master_inference(image, task, query):
    if image is None: return "Error: No Image", "", "final_metrics.png"

    # Specific prompting for VQA vs Captioning
    prompt = f"Question: {query} Answer:" if task == "VQA" else None
    inputs = processor(image, text=prompt, return_tensors="pt").to(device, torch.float16)

    with torch.inference_mode():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=30,
            num_beams=1,
            do_sample=False
        )

    decoded = processor.decode(generated_ids[0], skip_special_tokens=True).strip()

    if task == "VQA":
        answer = decoded.split("Answer:")[-1].strip()
        return "", answer, "final_metrics.png"
    else:
        return decoded, "", "final_metrics.png"

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🤖 Image Captioning + VQA via BLIP-2 ")
    with gr.Row():
        with gr.Column():
            input_img = gr.Image(type="pil", label="Input Image")
            task_type = gr.Radio(["Captioning", "VQA"], label="Task Selection", value="Captioning")
            vqa_query = gr.Textbox(label="Question (VQA only)")
            run_btn = gr.Button("🚀 Generate Result", variant="primary")
        with gr.Column():
            cap_out = gr.Textbox(label="Generated Caption")
            vqa_out = gr.Textbox(label="VQA Answer")
            metrics_plot = gr.Image(value="final_metrics.png", label="Training Progress")

    run_btn.click(master_inference, [input_img, task_type, vqa_query], [cap_out, vqa_out, metrics_plot])

demo.launch(share=True)